In [ ]:
def path_dataset(mouseMine, KOBAS_identify):
    import pandas as pd
    from tqdm import tqdm
    pd.options.mode.chained_assignment = None

    # Creating the lists of genes id
    ens_lists = KOBAS_identify['Input']
    mgi_lists = KOBAS_identify['Input_MGI']

    # Takes from mouseMine only the columns that I need and create a dataframe with all the pathways gruped per gene id
    mM = mouseMine[
        ['Allele > Ontology Annotations > Ontology Term . Identifier', 'Allele > Feature > Primary Identifier']]
    mM2 = mM.groupby(['Allele > Feature > Primary Identifier'], sort=False)

    # to speed up the search of the phenotypes related to each gene inside the lists, it's created a dictionary containing
    # a gene as key and list of phen_id related to it
    d = {}
    genes = mouseMine.drop_duplicates('Allele > Feature > Primary Identifier', inplace=False, ignore_index=True)
    genes = genes['Allele > Feature > Primary Identifier']

    for i in range(0, len(genes)):
        d[genes[i]] = mM['Allele > Ontology Annotations > Ontology Term . Identifier'][mM2.groups[genes[i]]].values


    # Create the subdataframe of each pathway, most time consuming step takes up to 1/2 min to complete
    df_list = []
    df = pd.DataFrame(columns=['MGI_ID', 'ENS_ID', 'Pheno_list'])
    for i in tqdm(range(0, len(ens_lists))):
        df.MGI_ID = (mgi_lists[i].translate({ord(i): None for i in '[]\'\n'})).split(',')
        df.MGI_ID = df.MGI_ID.str.strip()
        df.ENS_ID = (ens_lists[i].translate({ord(i): None for i in '[]\'\n'})).split(',')
        df.ENS_ID = df.ENS_ID.str.strip()
        for j in df.MGI_ID:
            if j in d.keys():
                df['Pheno_list'][df['MGI_ID'][df['MGI_ID'] == j].index[0]] = d[j]
        df_list.append(df)
        df = df.iloc[0:0]
    del (df, i)

    # Remove lines with Nan values (genes not found in the list)
    for i in range(0,len(df_list)):
        df_list[i] = df_list[i].dropna()
        df_list[i] = df_list[i].reset_index(drop=True)

    # Combining all the info into the final table

    path_data = pd.DataFrame(columns=['Pathway', 'Path_ID', 'Pheno_data'])
    path_data['Pathway'] = KOBAS_identify.Term
    path_data['Path_ID'] = KOBAS_identify.ID
    for i in range(0, len(df_list)):
        path_data['Pheno_data'][i] = pd.DataFrame(df_list[i])

    d = []

    #Removing the rows with only empty subdataframes and converting the Pheno_list fields from series to array
    for i in range(0, len(path_data)):
        if path_data.loc[i]['Pheno_data'].empty:
            d.append(i)
    path_data.drop(d, inplace=True)
    path_data.reset_index(drop=True, inplace=True)
    del (d, i)
    
    return path_data, df_list
